# Puan AI — Pipeline RAG EduPuan (parent-child)

Notebook ini menjalankan **seluruh sistem Puan AI** dari awal sampai akhir. Kodenya ada di folder `puan_rag/` dan `scripts/`;
notebook ini cukup "memencet tombol" tiap tahap sambil menjelaskan apa yang terjadi.

```
PDF peraturan ─► teks bersih ─► KNOWLEDGE BASE ─► INDEKS VEKTOR ─► (user bertanya) ─► tulis ulang ─► cari pasal ─► Gemini menjawab
                 (sudah jadi)    parent + child     embedding                          bahasa hukum    parent-child
```

**Urutan:** Bagian A (persiapan) → B (bangun data) → C (coba tanya) → D (evaluasi) → E (opsional: model e5) → F (lihat API).
Penjelasan lengkap tiap tahap ada di `docs/01_arsitektur_dan_alur.md`.

## A1. Hubungkan Google Drive dan buka folder proyek

**Yang kita suruh Colab lakukan:** menyambungkan Drive, lalu masuk ke folder `puan-ai`.
Kalau yang ada di Drive masih file `puan-ai.zip`, Colab akan membongkarnya dulu.

**Tujuannya:** semua hasil (knowledge base, indeks, hasil evaluasi) tersimpan di Drive, jadi tidak hilang saat Colab ditutup.

> Sebelum menjalankan: unggah `puan-ai.zip` ke folder `MyDrive/edupuan-rag/` (folder yang sama dengan notebook lama).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
INDUK = '/content/drive/MyDrive/edupuan-rag'          # folder proyek lama (berisi output/ dari notebook sebelumnya)
PROYEK = f'{INDUK}/puan-ai'

if not os.path.exists(PROYEK) and os.path.exists(f'{INDUK}/puan-ai.zip'):
    !unzip -q "{INDUK}/puan-ai.zip" -d "{INDUK}"
os.chdir(PROYEK)
print('Sekarang di:', os.getcwd())
!ls

Mounted at /content/drive
Sekarang di: /content/drive/MyDrive/edupuan-rag/puan-ai
api   docs	 output    README.md		requirements.txt  supabase
data  notebooks  puan_rag  requirements-e5.txt	scripts		  tests


## A2. Pasang paket dan kunci Gemini

**Yang kita suruh Colab lakukan:** memasang paket Python yang dibutuhkan (daftar di `requirements.txt`), lalu mengambil
kunci Gemini dari **Colab Secrets** (ikon kunci 🔑 di kiri, nama `GEMINI_API_KEY`, "Akses notebook" aktif).

**Tujuannya:** kunci tidak pernah tertulis di kode, jadi aman kalau notebook dibagikan.

**Hasil yang benar:** `Kunci Gemini terbaca.`

In [2]:
!pip install -q -r requirements.txt pytest
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('Kunci Gemini terbaca.' if os.environ['GEMINI_API_KEY'] else 'Kunci BELUM ada di Secrets!')

Kunci Gemini terbaca.


## A3. Uji otomatis (tanpa memakai kuota)

**Yang kita suruh Colab lakukan:** menjalankan 7 uji di `tests/test_puan.py` memakai komponen tiruan.

**Tujuannya:** memastikan semua bagian tersambung (knowledge base 248 parent / 694 child, pemecahan huruf, alur jawab, API)
**sebelum** memakai kuota Gemini. Kalau ada yang gagal, berhenti dulu dan cek pesannya.

**Hasil yang benar:** `7 passed`

In [3]:
!python -m pytest tests -q

.......                                                                  [100%]
7 passed in 16.21s


## B1. Bangun knowledge base parent-child

**Yang kita suruh Colab lakukan:** membaca teks bersih di `data/bersih/`, memotongnya jadi **parent** (pasal / angka SEOJK)
dan **child** (ayat / huruf / definisi), lalu memasang rujukan antar-pasal dan tautan antar-dokumen.
Dibuat dua versi: biasa, dan versi yang daftar panjangnya dipecah per huruf (untuk dibandingkan di bagian D).

**Tujuannya:** menyiapkan "rak buku" yang nanti dicari Puan AI. Pasal mana yang diambil diatur di `puan_rag/kurasi.py`.

**Hasil yang benar:** semua baris `OK`, lalu `248 parent | 889 child (694 dicari ...)` dan versi pecah `1028 child (833 dicari ...)`.

In [4]:
!python scripts/bangun_kb.py
!python scripts/bangun_kb.py --pecah-huruf

OK      | ID parent dobel: 0
OK      | ID child dobel: 0
OK      | child tanpa parent: 0
OK      | rujukan ke pasal yang tidak ada: 0
OK      | tautan TERKAIT yang salah kode: 0
OK      | parent tanpa child yang bisa dicari: 0

Knowledge base: 248 parent | 889 child (694 dicari, 195 ayat sanksi dilewati)
Parent per dokumen: {'POJK22': 80, 'POJK40': 23, 'UUPDP': 48, 'SEOJK19': 97}
Tersimpan di /content/drive/MyDrive/edupuan-rag/puan-ai/output/knowledge_base.json
OK      | ID parent dobel: 0
OK      | ID child dobel: 0
OK      | child tanpa parent: 0
OK      | rujukan ke pasal yang tidak ada: 0
OK      | tautan TERKAIT yang salah kode: 0
OK      | parent tanpa child yang bisa dicari: 0

Knowledge base (pecah huruf): 248 parent | 1028 child (833 dicari, 195 ayat sanksi dilewati)
Parent per dokumen: {'POJK22': 80, 'POJK40': 23, 'UUPDP': 48, 'SEOJK19': 97}
Tersimpan di /content/drive/MyDrive/edupuan-rag/puan-ai/output/knowledge_base_pecah.json


## B2. Intip isi knowledge base (bahan belajar)

**Yang kita suruh Colab lakukan:** menampilkan satu parent (POJK 22/2023 Pasal 62, etika penagihan) beserta child-nya.

**Tujuannya:** melihat langsung konsep parent-child: child kecil dipakai untuk **mencari**, parent utuh dipakai untuk **menjawab**.

In [5]:
import sys, json
sys.path.insert(0, PROYEK)
from puan_rag import knowledge_base as K
kb = K.muat('output/knowledge_base.json')
parent = next(p for p in kb['parents'] if p['id'] == 'POJK22:62')
print('PARENT', parent['id'], '|', K.label_sumber(parent))
print('Lokasi :', parent['lokasi'])
print('Terkait:', parent['terkait'])
print(parent['teks'][:600], '...\n')
for c in kb['children']:
    if c['parent_id'] == 'POJK22:62':
        print(f"  CHILD {c['id']:14} [{c['jenis']:6}] {c['teks'][:90]}")

PARENT POJK22:62 | POJK 22/2023 Pasal 62
Lokasi : POJK 22/2023 › BAB II KETENTUAN PELINDUNGAN KONSUMEN DAN MASYARAKAT DI SEKTOR JASA KEUANGAN › Bagian Kedelapan Pemberian Layanan atas Penggunaan Produk dan Layanan › Paragraf 2 Penagihan untuk Produk Kredit dan Pembiayaan oleh PUJK
Terkait: ['SEOJK19:VIII.1', 'SEOJK19:XX.9', 'UUPDP:65', 'UUPDP:67']
(1) PUJK wajib memastikan penagihan kredit atau pembiayaan kepada Konsumen dilaksanakan sesuai dengan norma yang berlaku di masyarakat dan ketentuan peraturan perundang-undangan.
(2) Dalam memastikan tindakan penagihan sebagaimana dimaksud pada ayat (1), PUJK wajib memastikan penagihan dilakukan:
a. tidak menggunakan cara ancaman, kekerasan dan/atau tindakan yang bersifat mempermalukan Konsumen;
b. tidak menggunakan tekanan secara fisik maupun verbal;
c. tidak kepada pihak selain Konsumen;
d. tidak secara terus menerus yang bersifat mengganggu;
e. di tempat alamat penagihan atau domisili Konsu ...

  CHILD POJK22:62.1    [isi   ] (1) PUJK waj

## B3. Buat indeks vektor Gemini

**Yang kita suruh Colab lakukan:** mengubah setiap child jadi deretan angka (vektor) dengan `gemini-embedding-001`.

- Perintah pertama **mengimpor embedding dari notebook lama** (`output/embeddings.npy`), jadi 694 child tidak perlu di-embed ulang.
- Perintah kedua membuat indeks versi pecah huruf: hanya **154 potongan baru** yang di-embed, sisanya dipakai ulang.

**Tujuannya:** menyiapkan pencarian berdasarkan makna, bukan sekadar kata yang sama.

**Hasil yang benar:** `694 dari 694 child diimpor ...` lalu `833 child dicari | 679 sudah punya vektor | 154 perlu di-embed`.
Kalau folder lama tidak ada, hapus bagian `--impor-lama ...`; semua child akan di-embed (± 7 menit).

In [6]:
!python scripts/buat_indeks.py --model gemini

694 child dicari | 0 sudah punya vektor | 694 perlu di-embed
   50/694 teks di-embed
   100/694 teks di-embed
embedding Gemini sibuk (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '), coba lagi dalam 60 detik...
   150/694 teks di-embed
   200/694 teks di-embed
embedding Gemini sibuk (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '), coba lagi dalam 60 detik...
   250/694 teks di-embed
   300/694 teks di-embed
embedding Gemini sibuk (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '), coba lagi dalam 60 detik...
   350/694 teks di-embed
   400/694 teks di-embed
embedding Gemini sibuk (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '), coba lagi dalam 60 detik...
   450/694 teks di-embed
   500/694 teks di-embed
embedding Gemini sibuk (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '), coba lagi dalam 60 detik...
   550/694 teks di-embed
   600/694 teks di-embed
embedding Gemini sibuk (429 RESOURCE_EXHAUSTED. {'error': {'code'

In [7]:
from google import genai
c = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
nama = [m.name for m in c.models.list()]
for n in nama:
    if 'flash' in n or 'embedding' in n:
        print(n)
print('\n--- Cek model yang dipakai Puan ---')
for cari in ['gemini-3.6-flash', 'gemini-3.1-flash-lite', 'gemini-flash-latest', 'gemini-embedding-001']:
    print('✅' if f'models/{cari}' in nama else '❌', cari)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview

--- Cek model yang dipakai Puan --

## C1. Coba tanya Puan AI

**Yang kita suruh Colab lakukan:** menjalankan seluruh alur untuk beberapa pesan: tulis ulang → cari pasal → Gemini menjawab.
Ditampilkan juga kueri hasil tulis ulang dan pasal yang terambil, supaya kelihatan "isi kepala" Puan AI.

**Tujuannya:** melihat hasil nyata dari setiap jenis pesan: bertanya, curhat, di luar topik, dan di luar cakupan.

In [8]:
import sys, json
sys.path.insert(0, PROYEK)
from puan_rag import PuanAI, Pengaturan, Pesan
puan = PuanAI.dari_folder(Pengaturan())          # bawaan: embedding gemini, tanpa pecah huruf

def coba(pesan, riwayat=None):
    h = puan.jawab(pesan, riwayat)
    print('=' * 90, '\nUSER  :', pesan)
    print(f'STATUS: {h.status} | krisis: {h.krisis} | skor tertinggi: {h.skor_tertinggi}')
    print('KUERI :', h.kueri)
    print('PASAL :', [s['label'] for s in h.sumber])
    print('-' * 90, '\n' + h.jawaban)
    return h

coba('DC nagih ke kantor dan bilang ke atasan aku, boleh gak sih?')
coba('aku capek banget, udah 3 bulan galbay di 4 pinjol, tiap hari diteror, malu sama keluarga')
coba('rekomendasi resep nasi goreng yang enak dong')
coba('kalau aku meninggal, utang pinjolku diwarisi ke anak gak?')

USER  : DC nagih ke kantor dan bilang ke atasan aku, boleh gak sih?
STATUS: pasal | krisis: False | skor tertinggi: 0.6792774200439453
KUERI : Apakah tindakan penagihan oleh pihak Penyelenggara LPBBTI kepada pihak selain Penerima Dana, yaitu atasan di kantor, diperbolehkan menurut ketentuan peraturan perundang-undangan?
PASAL : ['POJK 22/2023 Pasal 61', 'POJK 22/2023 Pasal 62', 'SEOJK 19/2025 angka romawi XI angka 1', 'POJK 22/2023 Pasal 60', 'SEOJK 19/2025 angka romawi XX angka 9']
------------------------------------------------------------------------------------------ 
Pasti rasanya malu dan stres banget ya ketika *debt collector* (DC) mendadak datang ke kantor dan bahkan melaporkan masalahmu ke atasan. 

Jawabannya: **sama sekali tidak boleh.** 

Perusahaan pinjol dilarang keras melakukan penagihan kepada pihak selain kamu, termasuk atasan, rekan kerja, atau keluarga. Aturan penagihan juga melarang penggunaan cara-cara yang bersifat mempermalukan konsumen. Penagihan seharusnya han

HasilJawaban(jawaban='Wajar banget kalau pikiran soal beban utang dan masa depan anak bikin kamu cemas. Terima kasih sudah mau terbuka membagikan kekhawatiran ini ya.\n\nUntuk pertanyaan spesifik mengenai apakah utang pinjol diwariskan ke anak jika peminjam meninggal dunia, jujur aku belum punya informasi yang cukup untuk menjawabnya dengan pasti.\n\nKarena persoalan ini berkaitan erat dengan hukum waris dan ketentuan perjanjian, aku sarankan kamu berkonsultasi langsung dengan Lembaga Bantuan Hukum (LBH) atau advokat di kotamu untuk mendapat penjelasan hukum yang tepat. Kamu juga bisa menanyakan prosedur atau aturan terkait ke **Kontak OJK 157** (Telepon 157 atau WhatsApp 081-157-157-157).\n\nKalau saat ini kamu sedang merasa pusing atau kewalahan mencatat utang yang ada, kamu bisa gunakan fitur **Rangkum Semua Utang** di EduPuan untuk membantumu merapikannya perlahan. Tetap semangat ya, aku di sini kalau ada hal lain yang ingin kamu ceritakan.', status='pasal', krisis=False, sumber=[{

## C2. Percakapan lanjutan (riwayat)

**Yang kita suruh Colab lakukan:** mengirim pertanyaan lanjutan ("terus aku harus lapor ke mana?") bersama riwayat pesan sebelumnya.

**Tujuannya:** memastikan Puan AI paham konteks percakapan, seperti yang nanti terjadi di web.

In [9]:
h1 = coba('dc nyebarin data utang aku ke semua kontak di hp')
riwayat = [Pesan('user', 'dc nyebarin data utang aku ke semua kontak di hp'), Pesan('puan', h1.jawaban)]
coba('terus aku harus lapor ke mana?', riwayat)

USER  : dc nyebarin data utang aku ke semua kontak di hp
STATUS: pasal | krisis: False | skor tertinggi: 0.6925690174102783
KUERI : Penyelenggara Layanan Pendanaan Bersama Berbasis Teknologi Informasi melakukan penyebarluasan Data Pribadi dan informasi utang Penerima Dana kepada pihak selain Konsumen melalui penagihan.
PASAL : ['POJK 22/2023 Pasal 61', 'SEOJK 19/2025 angka romawi VIII angka 1', 'SEOJK 19/2025 angka romawi XI angka 1', 'POJK 22/2023 Pasal 60', 'POJK 22/2023 Pasal 62']
------------------------------------------------------------------------------------------ 
Pasti kaget dan stres banget saat data utangmu disebar oleh *debt collector* ke semua kontak di HP. Perlakuan kasar seperti ini sama sekali tidak bisa dibenarkan.

Dalam aturan, perusahaan pinjol dilarang keras melakukan penagihan kepada pihak selain kamu, apalagi menyebarkan informasi utangmu ke kontak telepon untuk mempermalukan. Selain itu, aplikasi pinjol resmi sebenarnya hanya diizinkan mengakses kamera, lokasi

HasilJawaban(jawaban='Bingung harus mulai melapor dari mana saat mengalami hal ini adalah kondisi yang sangat wajar. Kamu tidak perlu khawatir, kamu punya hak untuk mengadukan tindakan penagihan tersebut.\n\nKamu bisa menyampaikan pengaduan resmi secara langsung ke **Otoritas Jasa Keuangan (OJK)** melalui Sistem Layanan Konsumen dan Masyarakat Terintegrasi di Sektor Jasa Keuangan. Setelah laporanmu masuk, perusahaan pinjol wajib menindaklanjuti pengaduan tersebut dan memberikan tanggapan balik. Nanti, kamu diberi waktu 10 hari kerja untuk merespons tanggapan dari pihak perusahaan pinjol di sistem tersebut.\n\nSebelum melapor, pastikan semua bukti pelanggaran sudah siap. Kamu bisa memanfaatkan fitur **Dokumentasi & Lapor** di EduPuan untuk membantu mengumpulkan dan merapikan bukti-bukti penagihan kasar yang kamu alami.\n\nDasar aturannya: POJK 22/2023 Pasal 96', status='pasal', krisis=False, sumber=[{'id': 'POJK22:79', 'label': 'POJK 22/2023 Pasal 79', 'peran': 'utama'}, {'id': 'POJK22:

## D1. Evaluasi pencarian: 50 soal uji

**Yang kita suruh Colab lakukan:** menguji pencarian pasal untuk 50 soal di `data/referensi/soal_uji_puan_ai.xlsx`
pada **dua versi** (biasa vs pecah huruf) dan beberapa pengaturan jumlah pasal (`--grid`).
`--kueri-dari` memakai kueri tulis ulang dari uji 1 notebook lama, jadi angkanya **bisa dibandingkan langsung** dengan uji 1–2.

**Tujuannya:** memutuskan pakai versi mana berdasarkan angka, bukan perasaan.

**Cara membaca:**
- Versi `gemini` tanpa pecah harus sama dengan uji 2: **Kena 87.5, MRR 0.80, Lintas semua dok 40**. Kalau beda, ada yang berubah, jadi cek dulu.
- Pilih versi pecah huruf **hanya kalau** Kena naik **dan** tidak ada soal yang tadinya kena jadi meleset (lihat sheet detail di Excel).
- Jangan pilih pengaturan yang skornya naik sedikit tapi "Rata2 bahan" melonjak.

Hasil lengkap: `output/hasil_evaluasi.xlsx`.

In [ ]:
!python scripts/evaluasi.py --model gemini --pecah 0 1 --grid --kueri-dari "{INDUK}/output/hasil_uji_1.xlsx"

## D2. Uji jawaban lengkap (dinilai manusia)

**Yang kita suruh Colab lakukan:** Puan AI menjawab lengkap 15 soal (10 di luar cakupan + 5 yang pernah meleset), lalu
jawabannya disimpan ke `output/uji_jawaban.xlsx` dengan kolom penilaian kosong.

**Tujuannya:** memeriksa hal yang tidak bisa dihitung mesin: nada hangat, ketepatan hukum, kejujuran saat tidak tahu, dan
**tidak ada pasal/angka karangan**. Penilaian sebaiknya dilakukan oleh orang hukum (kamu).

Memakai kuota `gemini-3.6-flash` sekitar 15 panggilan.

In [ ]:
!python scripts/uji_jawaban.py

## E. (Opsional) Bandingkan dengan model e5

**Yang kita suruh Colab lakukan:** memasang `sentence-transformers` (± 2–3 menit), membuat indeks dengan `multilingual-e5-small`
(berjalan di Colab, tanpa kuota), lalu mengevaluasi Gemini vs e5 dengan **kueri yang sama**.

**Tujuannya:** menjawab pertanyaan tim "lebih bagus Gemini atau e5?" dengan data. Perhatikan juga bagian **Kalibrasi ambang e5**:
ambang e5 di `puan_rag/config.py` (0.80) masih sementara, jadi ganti dengan "usulan ambang" kalau ada.

**Catatan:** e5 hanya membaca 512 token pertama tiap child, jadi chunk panjang terpotong. Itulah gunanya membandingkan juga versi pecah huruf.

In [ ]:
!pip install -q -r requirements-e5.txt
!python scripts/buat_indeks.py --model e5
!python scripts/buat_indeks.py --model e5 --pecah-huruf
!python scripts/evaluasi.py --model gemini e5 --pecah 0 1 --kueri-dari "{INDUK}/output/hasil_uji_1.xlsx"

## F. Lihat API seperti yang diterima web

**Yang kita suruh Colab lakukan:** menyalakan API FastAPI di dalam notebook (tanpa server sungguhan), lalu mengirim pesan
seperti yang nanti dilakukan Next.js.

**Tujuannya:** melihat bentuk data JSON yang akan dipakai tim frontend: `jawaban`, `status`, `krisis`, `sumber`.

In [10]:
from fastapi.testclient import TestClient
import api.main as api
with TestClient(api.app) as klien:
    print(klien.get('/health').json())
    r = klien.post('/chat', json={'pesan': 'bunga pinjol maksimal berapa per hari?', 'riwayat': []})
    print(r.status_code)
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))

{'status': 'ok', 'model_embedding': 'gemini', 'pecah_huruf': False, 'ambang': 0.6, 'jumlah_parent': 248, 'jumlah_child_dicari': 694, 'model_jawab': 'gemini-3.6-flash'}
200
{
  "jawaban": "Wajar banget kamu ingin tahu batas bunga pinjol supaya tidak terjebak biaya yang terlampau tinggi. Sini, Puan jelaskan aturan resminya.\n\nMenurut aturan OJK, total biaya pinjaman (bunga ditambah biaya administrasi atau biaya platform) disebut manfaat ekonomi. Batas maksimal per harinya yaitu:\n\n**1. Pinjaman Konsumtif (kebutuhan pribadi):**\n- Tenor sampai 6 bulan: maksimal **0,3% per hari**.\n- Tenor di atas 6 bulan: maksimal **0,2% per hari**.\n\n**2. Pinjaman Produktif (modal usaha):**\n- Pinjaman sampai Rp50 juta: maksimal **0,275% per hari** (tenor ≤ 6 bulan) atau **0,1% per hari** (tenor > 6 bulan).\n- Pinjaman di atas Rp50 juta: maksimal **0,1% per hari** (semua tenor).\n\nPersentase tersebut sudah mencakup bunga dan biaya administrasi lainnya yang dikenakan oleh perusahaan pinjol. \n\nSebelu

In [11]:
print(r.json()['jawaban'])

Wajar banget kamu ingin tahu batas bunga pinjol supaya tidak terjebak biaya yang terlampau tinggi. Sini, Puan jelaskan aturan resminya.

Menurut aturan OJK, total biaya pinjaman (bunga ditambah biaya administrasi atau biaya platform) disebut manfaat ekonomi. Batas maksimal per harinya yaitu:

**1. Pinjaman Konsumtif (kebutuhan pribadi):**
- Tenor sampai 6 bulan: maksimal **0,3% per hari**.
- Tenor di atas 6 bulan: maksimal **0,2% per hari**.

**2. Pinjaman Produktif (modal usaha):**
- Pinjaman sampai Rp50 juta: maksimal **0,275% per hari** (tenor ≤ 6 bulan) atau **0,1% per hari** (tenor > 6 bulan).
- Pinjaman di atas Rp50 juta: maksimal **0,1% per hari** (semua tenor).

Persentase tersebut sudah mencakup bunga dan biaya administrasi lainnya yang dikenakan oleh perusahaan pinjol. 

Sebelum setuju meminjam, pastikan kamu memeriksa rincian biayanya secara teliti. Kamu bisa coba gunakan fitur **Cek Penawaran** di EduPuan untuk membantu menghitung total biaya agar kamu tahu persis berapa ya

In [12]:
!python --version

Python 3.13.18


## Selesai — keputusan yang perlu diambil dari hasil notebook ini

1. **Versi knowledge base:** biasa atau pecah huruf? (D1). Isi `PUAN_PECAH_HURUF` di `.env` server sesuai keputusan.
2. **Model embedding:** Gemini atau e5? (E). Isi `PUAN_EMBED`.
3. **Jawaban:** ada pasal/angka karangan atau jawaban yang kurang tepat secara hukum? (D2). Catat, lalu perbaiki prompt/kurasi.
4. **Kunci soal 12 & 40:** perlu ditambah POJK22:77 dan POJK22:45? (keputusan hukum, lihat `docs/03_keputusan_dan_hasil_uji.md`).